In [2]:
import os 
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import ( 
        Dataset, 
        DataLoader, 
        random_split, 
        WeightedRandomSampler
    )
import torchvision.models as models 
import torchvision.transforms.functional as TF 
from torchvision.transforms import ColorJitter 
from PIL import Image 

SEED = 42 
random.seed(SEED) 
np.random.seed(SEED) 
torch.manual_seed(SEED) 

if torch.cuda.is_available(): 
    torch.cuda.manual_seed_all(SEED) 

torch.backends.cudnn.deterministic = True 
torch.backends.cudnn.bechnmark = False 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device : cuda
PyTorch: 2.11.0+cu128
Tesla T4


In [3]:
import os 
import random 
import torch
from PIL import Image 
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler 
import torchvision.transforms.functional as TF 
from torchvision.transforms import ColorJitter

SEED = 42 
random.seed(SEED) 
torch.manual_seed(SEED)
if torch.cuda.is_available(): 
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

class CAT2000Dataset(Dataset): 
    OFFICIAL_CATEGORIES = [
        "Action", "Affective", "Art", "Cartoon", "DigitalArt", 
        "Everyday", "Fractal", "Indoor", "Inverted", "LightNormal", 
        "LineDrawing", "Loud", "LowResolution", "Natural", "Object",
        "Outdoor", "Photo", "Random", "Social", "Synthetic" 
    ]
    
    def __init__(self, root_dir, augment=False, test_mode=False): 
        self.root_dir = os.path.abspath(root_dir)
        self.stimuli_dir = os.path.join(root_dir, "Stimuli")
        self.maps_dir = os.path.join(root_dir, "FixationMaps")
        self.test_mode = test_mode

        if test_mode:
            self.maps_dir = None
        else:
            self.maps_dir = os.path.join(root_dir, "FixationMaps")
        self.augment = augment 
        self.color_jitter = ColorJitter( 
            brightness = 0.15, 
            contrast = 0.15,  
            saturation = 0.10, 
            hue = 0.02
                                        )
        self.cat_to_idx = { 
            cat: idx 
            for idx, cat in enumerate(self.OFFICIAL_CATEGORIES)
            }
        self.samples = [] 
        self._index_dataset() 
        
    def _index_dataset(self): 
        for cat in self.OFFICIAL_CATEGORIES: 
            
            stim_folder = os.path.join(self.stimuli_dir, cat)
            
            if not os.path.exists(stim_folder): 
                continue
            
            if self.test_mode: 
                map_folder = os.path.join(stim_folder, "Output")
            else: 
                map_folder = os.path.join(self.maps_dir, cat)
            
            for filename in sorted(os.listdir(stim_folder)): 
                if not filename.lower().endswith((".jpg", ".jpeg", ".png")): 
                    continue
                
                image_path = os.path.join(stim_folder, filename) 
                
                if self.test_mode:
                    stem = os.path.splitext(filename)[0]
                    map_path = os.path.join(
                        map_folder,
                        f"{stem}_SaliencyMap.jpg"
                    )
                else:
                    map_path = os.path.join(map_folder, filename)

                if os.path.exists(map_path):
                    self.samples.append({
                        "image": image_path,
                        "map": map_path,
                        "cat_idx": self.cat_to_idx[cat]
                    })
        
    def __len__(self): 
        return len(self.samples)
    
    def __getitem__(self, idx): 
        sample = self.samples[idx] 
        image = Image.open(sample["image"]).convert("RGB")
        fix_map = Image.open(sample["map"]).convert("L")
        
        if self.augment: 
            if random.random() < 0.5: 
                image = TF.hflip(image)
                fix_map = TF.hflip(fix_map)
                
            image = self.color_jitter(image)
        
        image = TF.resize(
            image,
            (224, 224),
            interpolation=TF.InterpolationMode.BILINEAR
        )
        fix_map = TF.resize( 
            fix_map, 
            (224, 224), 
            interpolation=TF.InterpolationMode.BILINEAR
            )
        
        image_tensor = TF.to_tensor(image)
        fix_tensor = TF.to_tensor(fix_map)
        
        fix_tensor = (
        fix_tensor - fix_tensor.min()
        ) / (
        fix_tensor.max() - fix_tensor.min() + 1e-6
        )
    
        image_tensor = TF.normalize( 
                image_tensor, 
                mean=[0.485, 0.456, 0.406], 
                std = [0.229, 0.224, 0.225])
    
        return image_tensor, fix_tensor, sample["cat_idx"]
    
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)
    
base_path = "/content/drive/MyDrive/CAT2000/trainSet"
full_dataset = CAT2000Dataset( 
            root_dir=base_path, 
            augment=True
            )
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size 

generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset = random_split( 
                full_dataset,  
                [train_size, val_size], 
                generator= generator
                )
train_labels = [
    full_dataset.samples[i]["cat_idx"]
    for i in train_dataset.indices
]
class_counts = torch.bincount(torch.tensor(train_labels, dtype=torch.long))
class_weights = 1.0 / class_counts.float() 

sample_weights = [ 
        class_weights[label] 
        for label in train_labels
]
sampler = WeightedRandomSampler( 
            sample_weights, 
            num_samples = len(sample_weights),
            replacement=True
        )

production_loader = DataLoader( 
        train_dataset, 
        batch_size=8, 
        sampler= sampler,
        worker_init_fn=seed_worker
    )
validation_loader = DataLoader( 
        val_dataset, 
        batch_size=8, 
        shuffle=False,
        worker_init_fn=seed_worker
)

print(f"Total images      : {len(full_dataset)}")
print(f"Training images   : {len(train_dataset)}")
print(f"Validation images : {len(val_dataset)}")
print(f"Categories        : {len(full_dataset.OFFICIAL_CATEGORIES)}")
print(f"Random seed      : {SEED}")


Total images      : 755
Training images   : 604
Validation images : 151
Categories        : 20
Random seed      : 42


In [4]:
import torch.nn as nn 
import torchvision.models as models 

backbone = models.resnet50(weights=None)
modules = list(backbone.children())[:-2]
feature_extractor = nn.Sequential(*modules).to(device)
feature_extractor.train() 
print("--> Scratch ResNet-50 initialized successfully.")
print("--> Backbone parameters are trainable.")

--> Scratch ResNet-50 initialized successfully.
--> Backbone parameters are trainable.


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SaliencyDecoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(2048, 512, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(512, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):

        x = F.relu(self.conv1(x))

        x = F.interpolate(
            x,
            scale_factor=4,
            mode="bilinear",
            align_corners=False
        )

        x = F.relu(self.conv2(x))

        x = F.interpolate(
            x,
            scale_factor=4,
            mode="bilinear",
            align_corners=False
        )

        x = self.conv3(x)

        x = F.interpolate(
            x,
            scale_factor=2,
            mode="bilinear",
            align_corners=False
        )

        return torch.sigmoid(x)

decoder = SaliencyDecoder().to(device)

print("--> Saliency decoder initialized.")

--> Saliency decoder initialized.


In [6]:
import torch

def compute_cc(pred, target): 
    pred = pred - pred.mean(dim=(2,3), keepdim=True)
    target = target - target.mean(dim=(2,3), keepdim=True)
    
    numerator = (pred * target).sum(dim=(2,3))
    
    denominator = (torch.sqrt((pred**2).sum(dim=(2,3))) * torch.sqrt((target**2).sum(dim=(2,3))) + 1e-8)
    
    return (numerator / denominator).mean() 

def compute_sim(pred, target): 
    pred = pred / (pred.sum(dim=(2,3), keepdim=True) + 1e-8)
    target = target / (target.sum(dim=(2,3), keepdim=True) + 1e-8)
    
    return torch.min(pred, target).sum(dim=(2,3)).mean() 

def reduce_feature_dimension(features, n_components=128): 
    B, C, H, W = features.shape 
    
    projection = torch.randn(C, n_components, device=features.device)
    projection = torch.nn.functional.normalize(projection, dim=0)
    reshaped = features.permute(0,2,3,1).reshape(-1,C)
    reduced = reshaped @ projection
    reduced = reduced.reshape(B, H, W, n_components) 
    reduced = reduced.permute(0, 3, 1, 2)
    
    return reduced


In [7]:
import torch

def compute_covariance_manifold(features):

    # Check original backbone features
    print("=" * 60)
    print("Original features")
    print("Shape:", features.shape)
    print("NaN:", torch.isnan(features).any().item())
    print("Inf:", torch.isinf(features).any().item())
    print("Min:", features.min().item())
    print("Max:", features.max().item())

    # Reduce dimensionality
    features = reduce_feature_dimension(features, n_components=48)

    # Check reduced features
    print("\nReduced features")
    print("Shape:", features.shape)
    print("NaN:", torch.isnan(features).any().item())
    print("Inf:", torch.isinf(features).any().item())
    print("Min:", features.min().item())
    print("Max:", features.max().item())

    B, C, H, W = features.shape
    N = H * W

    print("\nH*W =", N)

    reshaped = features.reshape(B, C, N)

    mean = reshaped.mean(dim=2, keepdim=True)
    centered = reshaped - mean

    print("\nCentered features")
    print("NaN:", torch.isnan(centered).any().item())
    print("Inf:", torch.isinf(centered).any().item())

    covariance = torch.bmm(
        centered,
        centered.transpose(1, 2)
    ) / (N - 1)

    print("\nAfter covariance")
    print("NaN:", torch.isnan(covariance).any().item())
    print("Inf:", torch.isinf(covariance).any().item())

    covariance = 0.5 * (
        covariance + covariance.transpose(1, 2)
    )

    covariance = covariance + 1e-4 * torch.eye(
        C,
        device=features.device
    ).unsqueeze(0)

    print("\nFinal covariance")
    print("NaN:", torch.isnan(covariance).any().item())
    print("Inf:", torch.isinf(covariance).any().item())
    print("Min:", features.min().item())
    print("Max:", features.max().item())
    print("=" * 60)

    return covariance
def matrix_logarithm(spd_matrix):
    try:
        eigenvalues, eigenvectors = torch.linalg.eigh(spd_matrix)
    except RuntimeError:
        spd_matrix = spd_matrix + 1e-3 * torch.eye(
        spd_matrix.shape[-1],
        device=spd_matrix.device
        ).unsqueeze(0)
    print("SPD shape:", spd_matrix.shape)
    print("Symmetry error:", torch.norm(spd_matrix - spd_matrix.transpose(-1, -2)).item())
    eigenvalues, eigenvectors = torch.linalg.eigh(spd_matrix)
    eigenvalues = torch.clamp(
        eigenvalues,
        min=1e-6
    )
    log_eigenvalues = torch.log(eigenvalues)

    log_diag = torch.diag_embed(log_eigenvalues)
    return (
        eigenvectors
        @ log_diag
        @ eigenvectors.transpose(1, 2)
    )
def riemannian_smoothness_loss(features):
    spd = compute_covariance_manifold(features)
    tangent = matrix_logarithm(spd)
    tangent_mean = tangent.mean(
        dim=0,
        keepdim=True
    )
    return (
        (tangent - tangent_mean) ** 2
    ).mean()

In [8]:
images, fixation_maps, _ = next(iter(production_loader))

images = images.to(device)
fixation_maps = fixation_maps.to(device)

feature_extractor.train()
decoder.train()

features = feature_extractor(images)
predictions = decoder(features)

print("Features:", features.shape)
print("Predictions:", predictions.shape)
print("Ground Truth:", fixation_maps.shape)

cc = compute_cc(predictions, fixation_maps)
sim = compute_sim(predictions, fixation_maps)
smoothness = riemannian_smoothness_loss(features)

loss = (1 - cc) + 0.05 * smoothness

print(f"CC: {cc:.4f}")
print(f"SIM: {sim:.4f}")
print(f"Smoothness: {smoothness:.6f}")
print(f"Loss: {loss:.4f}")

loss.backward()

print("Forward and backward pass successful.")

Features: torch.Size([8, 2048, 7, 7])
Predictions: torch.Size([8, 1, 224, 224])
Ground Truth: torch.Size([8, 1, 224, 224])
Original features
Shape: torch.Size([8, 2048, 7, 7])
NaN: False
Inf: False
Min: 0.0
Max: 26.729934692382812

Reduced features
Shape: torch.Size([8, 48, 7, 7])
NaN: False
Inf: False
Min: -14.283698081970215
Max: 15.646967887878418

H*W = 49

Centered features
NaN: False
Inf: False

After covariance
NaN: False
Inf: False

Final covariance
NaN: False
Inf: False
Min: -14.283698081970215
Max: 15.646967887878418
SPD shape: torch.Size([8, 48, 48])
Symmetry error: 0.0
CC: 0.2606
SIM: 0.3346
Smoothness: 0.082695
Loss: 0.7435
Forward and backward pass successful.


In [9]:
print(sum(p.numel() for p in feature_extractor.parameters() if p.requires_grad))

23508032


In [10]:
import torch.optim as optim
import numpy as np

optimizer = optim.Adam(
    list(feature_extractor.parameters()) +
    list(decoder.parameters()),
    lr=1e-4
)

NUM_EPOCHS = 10
ALPHA = 0.05

best_cc = -float("inf")

train_history = {
    "loss": [],
    "cc": [],
    "sim": [],
    "smoothness": [],
    "val_loss": [],
    "val_cc": [],
    "val_sim": []
}

print("--> Starting training cycle...")

for epoch in range(NUM_EPOCHS):

    feature_extractor.train()
    decoder.train()

    epoch_total_loss = 0.0
    epoch_cc = 0.0
    epoch_sim = 0.0
    epoch_smooth = 0.0

    for images, fixation_maps, _ in production_loader:

        images = images.to(device)
        fixation_maps = fixation_maps.to(device)

        optimizer.zero_grad()

        features = feature_extractor(images)
        predicted_saliency = decoder(features)

        cc = compute_cc(predicted_saliency, fixation_maps)
        sim = compute_sim(predicted_saliency, fixation_maps)
        smoothness = riemannian_smoothness_loss(features)

        cc_loss = 1 - cc

        total_loss = cc_loss + ALPHA * smoothness

        total_loss.backward()

        optimizer.step()

        epoch_total_loss += total_loss.item()
        epoch_cc += cc.item()
        epoch_sim += sim.item()
        epoch_smooth += smoothness.item()

    avg_loss = epoch_total_loss / len(production_loader)
    avg_cc = epoch_cc / len(production_loader)
    avg_sim = epoch_sim / len(production_loader)
    avg_smooth = epoch_smooth / len(production_loader)

    feature_extractor.eval()
    decoder.eval()

    val_loss = 0.0
    val_cc = 0.0
    val_sim = 0.0

    with torch.no_grad():

        for images, fixation_maps, _ in validation_loader:

            images = images.to(device)
            fixation_maps = fixation_maps.to(device)

            features = feature_extractor(images)
            predicted_saliency = decoder(features)

            cc = compute_cc(predicted_saliency, fixation_maps)
            sim = compute_sim(predicted_saliency, fixation_maps)
            smoothness = riemannian_smoothness_loss(features)

            loss = (1 - cc) + ALPHA * smoothness

            val_loss += loss.item()
            val_cc += cc.item()
            val_sim += sim.item()

    avg_val_loss = val_loss / len(validation_loader)
    avg_val_cc = val_cc / len(validation_loader)
    avg_val_sim = val_sim / len(validation_loader)

    train_history["loss"].append(avg_loss)
    train_history["cc"].append(avg_cc)
    train_history["sim"].append(avg_sim)
    train_history["smoothness"].append(avg_smooth)

    train_history["val_loss"].append(avg_val_loss)
    train_history["val_cc"].append(avg_val_cc)
    train_history["val_sim"].append(avg_val_sim)

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {avg_loss:.4f} | "
        f"Train CC: {avg_cc:.4f} | "
        f"Train SIM: {avg_sim:.4f} | "
        f"Smoothness: {avg_smooth:.6f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val CC: {avg_val_cc:.4f} | "
        f"Val SIM: {avg_val_sim:.4f}"
    )

    if avg_val_cc > best_cc:

        best_cc = avg_val_cc

        torch.save(
            {
                "backbone": feature_extractor.state_dict(),
                "decoder": decoder.state_dict(),
            },
            "/content/drive/MyDrive/CAT2000/best_scratch_model.pth"
        )

np.save(
    "/content/drive/MyDrive/CAT2000/scratch_train_history.npy",
    train_history
)

print("\n--> Training Complete.")
print(f"Best Validation CC: {best_cc:.4f}")

--> Starting training cycle...
Original features
Shape: torch.Size([8, 2048, 7, 7])
NaN: False
Inf: False
Min: 0.0
Max: 21.327281951904297

Reduced features
Shape: torch.Size([8, 48, 7, 7])
NaN: False
Inf: False
Min: -10.217628479003906
Max: 12.03462028503418

H*W = 49

Centered features
NaN: False
Inf: False

After covariance
NaN: False
Inf: False

Final covariance
NaN: False
Inf: False
Min: -10.217628479003906
Max: 12.03462028503418
SPD shape: torch.Size([8, 48, 48])
Symmetry error: 0.0
Original features
Shape: torch.Size([8, 2048, 7, 7])
NaN: False
Inf: False
Min: 0.0
Max: 24.057964324951172

Reduced features
Shape: torch.Size([8, 48, 7, 7])
NaN: False
Inf: False
Min: -13.144859313964844
Max: 12.939364433288574

H*W = 49

Centered features
NaN: False
Inf: False

After covariance
NaN: False
Inf: False

Final covariance
NaN: False
Inf: False
Min: -13.144859313964844
Max: 12.939364433288574
SPD shape: torch.Size([8, 48, 48])
Symmetry error: 0.0
Original features
Shape: torch.Size([8, 2

In [11]:
analysis_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    worker_init_fn=seed_worker,
    pin_memory=torch.cuda.is_available()
)

checkpoint = torch.load(
    "/content/drive/MyDrive/CAT2000/best_scratch_model.pth",
    map_location=device
)

feature_extractor.load_state_dict(checkpoint["backbone"])
decoder.load_state_dict(checkpoint["decoder"])

feature_extractor.eval()
decoder.eval()

all_features = []
all_predictions = []
all_ground_truth = []
all_categories = []

print("Running inference on validation set...\n")

with torch.no_grad():

    for images, fixation_maps, categories in analysis_loader:

        images = images.to(device)

        features = feature_extractor(images)

        predictions = decoder(features)

        all_features.append(features.cpu())
        all_predictions.append(predictions.cpu())
        all_ground_truth.append(fixation_maps)
        all_categories.append(categories)

all_features = torch.cat(all_features, dim=0)
all_predictions = torch.cat(all_predictions, dim=0)
all_ground_truth = torch.cat(all_ground_truth, dim=0)
all_categories = torch.cat(all_categories, dim=0)

print("Inference complete.\n")

print(f"Features     : {all_features.shape}")
print(f"Predictions  : {all_predictions.shape}")
print(f"Ground Truth : {all_ground_truth.shape}")
print(f"Categories   : {all_categories.shape}")

torch.save(
    {
        "features": all_features,
        "predictions": all_predictions,
        "ground_truth": all_ground_truth,
        "categories": all_categories
    },
    "/content/drive/MyDrive/CAT2000/scratch_inference.pt"
)

Running inference on validation set...

Inference complete.

Features     : torch.Size([151, 2048, 7, 7])
Predictions  : torch.Size([151, 1, 224, 224])
Ground Truth : torch.Size([151, 1, 224, 224])
Categories   : torch.Size([151])


In [14]:
import torch
import numpy as np 

def compute_channel_entropy(features): 
    B, C, H, W = features.shape 
    entropy = [] 
    for i in range(B): 
        
        feat = features[i] 
        channel_energy = feat.abs().mean(dim=(1,2))
        probs = channel_energy / (channel_energy.sum() + 1e-12)
        H_channel = -(probs * torch.log2(probs + 1e-12)).sum() 
        entropy.append(H_channel.item())
        
    return np.array(entropy)

def compute_spatial_correlation(features):

    features = reduce_feature_dimension(features, n_components=48)

    B, C, H, W = features.shape
    correlations = []

    for i in range(B):

        feat = features[i].reshape(C, -1)
        feat = feat - feat.mean(dim=1, keepdim=True)
        std = feat.std(dim=1, keepdim=True)
        valid = (std.squeeze() > 1e-6)
        feat = feat[valid]
        if feat.shape[0] < 2:
            correlations.append(0.0)
            continue
        feat = feat / (feat.std(dim=1, keepdim=True) + 1e-8)
        corr = feat @ feat.T
        corr /= (feat.shape[1] - 1)
        mask = ~torch.eye(
            corr.shape[0],
            dtype=torch.bool,
            device=corr.device
        )

        correlations.append(
            corr[mask].abs().mean().item()
        )

    return np.array(correlations)

def compute_manifold_smoothness(features):

    spd = compute_covariance_manifold(features)

    tangent = matrix_logarithm(spd)

    tangent_mean = tangent.mean(dim=0, keepdim=True)

    smoothness = (
        (tangent - tangent_mean) ** 2
    ).mean(dim=(1,2))

    return smoothness.cpu().numpy()

def analyze_feature_tensor(features):

    results = {}

    results["entropy"] = compute_channel_entropy(features)

    results["spatial_corr"] = compute_spatial_correlation(features)

    results["smoothness"] = compute_manifold_smoothness(features)

    return results

In [15]:
import torch
import numpy as np

results = analyze_feature_tensor(all_features)

print("Analysis complete.\n")

for metric, values in results.items():

    print(
        f"{metric:15s}"
        f"Mean = {values.mean():.6f}   "
        f"Std = {values.std():.6f}"
    )

Original features
Shape: torch.Size([151, 2048, 7, 7])
NaN: False
Inf: False
Min: 0.0
Max: 44.23337936401367

Reduced features
Shape: torch.Size([151, 48, 7, 7])
NaN: False
Inf: False
Min: -15.330894470214844
Max: 19.532146453857422

H*W = 49

Centered features
NaN: False
Inf: False

After covariance
NaN: False
Inf: False

Final covariance
NaN: False
Inf: False
Min: -15.330894470214844
Max: 19.532146453857422
SPD shape: torch.Size([151, 48, 48])
Symmetry error: 0.0
Analysis complete.

entropy        Mean = 10.946608   Std = 0.074024
spatial_corr   Mean = 0.183868   Std = 0.013185
smoothness     Mean = 0.087128   Std = 0.014483


In [16]:
import os
import numpy as np

analysis_dir = "/content/drive/MyDrive/CAT2000/analysis"

os.makedirs(
    analysis_dir,
    exist_ok=True
)

np.save(
    os.path.join(
        analysis_dir,
        "scratch_analysis.npy"
    ),
    results,
    allow_pickle=True
)

print("Scratch analysis saved successfully.")

Scratch analysis saved successfully.
